In [1]:

# =============================================================================
# [CELL 0] Imports and helpers
# =============================================================================

import os
import re
import unicodedata
from pathlib import Path
from typing import List, Optional, Dict, Tuple
from collections import defaultdict

import numpy as np
import pandas as pd

os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

# Optional: avoid oversubscription
os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("MKL_NUM_THREADS", "1")
os.environ.setdefault("OPENBLAS_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("NUMEXPR_NUM_THREADS", "1")


def l2_normalize(v: np.ndarray) -> np.ndarray:
    n = float(np.linalg.norm(v))
    return (v / (n + 1e-12)).astype(np.float32)

def l2_normalize_rows(M: np.ndarray) -> np.ndarray:
    """Row wise L2 normalization for 2D arrays."""
    M = M.astype(np.float32, copy=False)
    denom = np.linalg.norm(M, axis=1, keepdims=True) + 1e-12
    return (M / denom).astype(np.float32)


def power_normalize(v: np.ndarray, p: float = 0.5) -> np.ndarray:
    """Power normalization: sign(v) * |v|^p."""
    v = v.astype(np.float32, copy=False)
    return (np.sign(v) * (np.abs(v) ** float(p))).astype(np.float32)




def power_normalize(v: np.ndarray, p: float = 0.5) -> np.ndarray:
    """Power normalization: sign(v) * |v|^p, then returns float32."""
    v = v.astype(np.float32, copy=False)
    return (np.sign(v) * (np.abs(v) ** float(p))).astype(np.float32)


def ensure_run(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    if "qid" not in df.columns or "docno" not in df.columns:
        raise ValueError("Run must contain columns: qid, docno")
    df["qid"] = df["qid"].astype(str)
    df["docno"] = df["docno"].astype(str)

    if "score" not in df.columns:
        df["score"] = 0.0
    df["score"] = df["score"].astype(float)

    if "rank" not in df.columns:
        df = df.sort_values(["qid", "score"], ascending=[True, False])
        df["rank"] = df.groupby("qid").cumcount() + 1

    return df


# =============================================================================
# [CELL 1] Paths and settings
# =============================================================================

# Inputs you provide
QB_PATH = "/Users/user/question-retrieval-KIPerWeb/testbeds/open-datafold.csv"
QUERIES_PATH = "/Users/user/question-retrieval-KIPerWeb/testbeds/queries_experiments/queries/queries.csv"
QRELS_PATH = "qrels_llm.txt"  # relative or absolute

# ESCO files
ESCO_DIR = "/Users/user/Downloads/ESCO"
SKILLS_PATH = f"{ESCO_DIR}/skills_de.csv"
REL_PATH = f"{ESCO_DIR}/skillSkillRelations_de.csv"
HIER_PATH = f"{ESCO_DIR}/skillsHierarchy_de.csv"

# Artifacts (robust absolute paths)
from pathlib import Path
import os

PROJECT_ROOT = Path("/Users/user/Projects/question-retrieval-KG").resolve()
ART_DIR = (PROJECT_ROOT / "var" / "artifacts" / "kg_reranker_v3_refactored").resolve()
ART_DIR.mkdir(parents=True, exist_ok=True)

INDEX_DIR = (ART_DIR / "terrier_index").resolve()
INDEX_DIR.mkdir(parents=True, exist_ok=True)

COMPLEX_CACHE_DIR = (ART_DIR / "pykeen_complex_cache").resolve()
COMPLEX_CACHE_DIR.mkdir(parents=True, exist_ok=True)

print("cwd:", os.getcwd())
print("INDEX_DIR:", INDEX_DIR)


# Retrieval models
# DENSE_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
DENSE_MODEL = "deutsche-telekom/gbert-large-paraphrase-cosine"
CE_MODEL = "cross-encoder/msmarco-MiniLM-L6-en-de-v1"

# ESCO skill retrieval settings

ESCO_MODEL = "sentence-transformers/paraphrase-multilingual-mpnet-base-v2"
ESCO_TOPK_DOC = 15
ESCO_TOPK_QUERY = 20
ESCO_MIN_SIM = 0.45

# Graph augmentation (this was used in your notebook for SVD, now applied to ComplEx too)
KG_EXPAND_HOPS = 1
KG_NEIGHBOR_W = 0.30

# Cutoffs
K_CAND = 100
K_FINAL = 50

# ComplEx training settings
COMPLEX_EMB_DIM = 512
COMPLEX_EPOCHS = 300
COMPLEX_BATCH_SIZE = 1024
COMPLEX_LR = 1e-3
COMPLEX_DEVICE = "cpu"  # set to "cuda" if available


# RotatE training settings (CPU)
ROTATE_EMB_DIM = 256
ROTATE_EPOCHS = 300
ROTATE_BATCH_SIZE = 1024
ROTATE_LR = 1e-3
ROTATE_DEVICE = "cpu"



# =============================================================================
# [CELL 2] Load dataset and queries, normalize text
# =============================================================================

_ws_re = re.compile(r"\s+")
_tags_re = re.compile(r"<[^>]+>")


def normalize_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)
    s = _tags_re.sub(" ", s)
    s = unicodedata.normalize("NFKC", s)
    s = s.lower()
    s = re.sub(r"[^0-9a-zA-ZäöüßÄÖÜẞ]+", " ", s)
    s = _ws_re.sub(" ", s).strip()
    return s


def safe_concat(parts: List[Optional[str]]) -> str:
    parts = [p for p in parts if p is not None]
    clean = []
    for p in parts:
        p = str(p)
        if p.strip() in ("", "N/A", "nan"):
            continue
        clean.append(p)
    return " ".join(clean)


def build_doc_text(row: pd.Series) -> str:
    # expects columns similar to your previous notebook
    return safe_concat([row.get("question"), row.get("choices_processed")])


qb = pd.read_csv(QB_PATH).fillna("N/A")
queries_file = pd.read_csv(QUERIES_PATH).fillna("")

if "test_item_id" not in qb.columns:
    raise ValueError("Dataset must contain column: test_item_id")
if "qid" not in queries_file.columns or "queries" not in queries_file.columns:
    raise ValueError("Queries must contain columns: qid, queries")

qb["docno"] = qb["test_item_id"].astype(str)
qb["raw_text"] = qb.apply(build_doc_text, axis=1)
qb["text"] = qb["raw_text"].map(normalize_text)

corpus = qb[["docno", "text"]].copy()

topics = queries_file.rename(columns={"queries": "query"})[["qid", "query"]].copy()
topics["qid"] = topics["qid"].astype(str)
topics["query"] = topics["query"].astype(str).map(normalize_text)

print("corpus:", corpus.shape, "topics:", topics.shape)


# =============================================================================
# [CELL 3] ESCO load, adjacency, and labels
# =============================================================================

skills_df = pd.read_csv(SKILLS_PATH).fillna("")
need = {"conceptUri", "preferredLabel"}
if not need.issubset(skills_df.columns):
    raise ValueError(f"ESCO skills must include {sorted(need)}")

skills_df["conceptUri"] = skills_df["conceptUri"].astype(str).str.strip()
skills_df["preferredLabel"] = skills_df["preferredLabel"].astype(str).str.strip()

uri2label = dict(zip(skills_df["conceptUri"], skills_df["preferredLabel"]))

rel_df = pd.read_csv(REL_PATH).fillna("")
hier_df = pd.read_csv(HIER_PATH).fillna("")

adj: Dict[str, set] = defaultdict(set)

# Optional relations
if "relationType" in rel_df.columns:
    rel_opt = rel_df[rel_df["relationType"].astype(str) == "optional"].copy()
else:
    rel_opt = rel_df.copy()

if "originalSkillUri" in rel_opt.columns and "relatedSkillUri" in rel_opt.columns:
    for a, b in zip(rel_opt["originalSkillUri"].astype(str), rel_opt["relatedSkillUri"].astype(str)):
        a = a.strip()
        b = b.strip()
        if a and b and a != "nan" and b != "nan":
            adj[a].add(b)
            adj[b].add(a)

# Hierarchy edges
lvl_cols = [c for c in ["Level 0 URI", "Level 1 URI", "Level 2 URI", "Level 3 URI"] if c in hier_df.columns]
for _, r in hier_df[lvl_cols].iterrows():
    uris = [str(x).strip() for x in r.tolist() if str(x).strip() and str(x).strip() != "nan"]
    for u, v in zip(uris[:-1], uris[1:]):
        adj[u].add(v)
        adj[v].add(u)

print("ESCO skills:", len(uri2label), "adj nodes:", len(adj))


# =============================================================================
# [CELL 4] ESCO ANN index (SentenceTransformer + FAISS)
# =============================================================================

import faiss
from sentence_transformers import SentenceTransformer

skills_nn = skills_df[["conceptUri", "preferredLabel"]].drop_duplicates("conceptUri").copy()
skills_nn = skills_nn[(skills_nn["conceptUri"] != "") & (skills_nn["conceptUri"] != "nan")]
skills_nn = skills_nn[(skills_nn["preferredLabel"] != "") & (skills_nn["preferredLabel"] != "nan")].reset_index(drop=True)

skill_vocab = skills_nn["conceptUri"].tolist()
skill_labels = skills_nn["preferredLabel"].tolist()

st_esco = SentenceTransformer(ESCO_MODEL)

X = st_esco.encode(
    skill_labels,
    convert_to_numpy=True,
    normalize_embeddings=True,
    show_progress_bar=True,
).astype("float32")

index_esco = faiss.IndexFlatIP(X.shape[1])
index_esco.add(X)

print("skills indexed:", len(skill_vocab), "dim:", X.shape[1])


def top_skills(texts: List[str], topk: int) -> List[List[Tuple[str, float]]]:
    Q = st_esco.encode(
        [str(t) for t in texts],
        convert_to_numpy=True,
        normalize_embeddings=True,
        show_progress_bar=False,
    ).astype("float32")
    scores, idxs = index_esco.search(Q, int(topk))

    out: List[List[Tuple[str, float]]] = []
    for i in range(len(texts)):
        pairs: List[Tuple[str, float]] = []
        for j, sc in zip(idxs[i], scores[i]):
            if j < 0:
                continue
            pairs.append((skill_vocab[j], float(sc)))
        out.append(pairs)
    return out


# =============================================================================
# [CELL 5] Extract ESCO skills for docs and queries
# =============================================================================

def pairs_to_df(ids: List[str], pairs: List[List[Tuple[str, float]]], id_col: str) -> pd.DataFrame:
    rows = []
    for _id, ps in zip(ids, pairs):
        for uri, sc in ps:
            if float(sc) < float(ESCO_MIN_SIM):
                continue
            rows.append({id_col: str(_id), "skill_uri": str(uri), "w": float(sc)})
    df = pd.DataFrame(rows)
    if df.empty:
        df = pd.DataFrame(columns=[id_col, "skill_uri", "w"])
    return df


doc_pairs = top_skills(corpus["text"].astype(str).tolist(), topk=ESCO_TOPK_DOC)
doc_skills_df = pairs_to_df(corpus["docno"].astype(str).tolist(), doc_pairs, id_col="docno")

qid_pairs = top_skills(topics["query"].astype(str).tolist(), topk=ESCO_TOPK_QUERY)
qid_skills_df = pairs_to_df(topics["qid"].astype(str).tolist(), qid_pairs, id_col="qid")

print("doc_skills_df:", doc_skills_df.shape, "unique docs:", doc_skills_df["docno"].nunique())
print("qid_skills_df:", qid_skills_df.shape, "unique qids:", qid_skills_df["qid"].nunique())


# =============================================================================
# [CELL 6] Graph augmentation (apply to ComplEx too)
# =============================================================================

def expand_pairs(skills_df_w: pd.DataFrame, id_col: str, hops: int, neighbor_w: float) -> pd.DataFrame:
    if skills_df_w.empty:
        return skills_df_w.copy()

    base = skills_df_w.copy()
    base[id_col] = base[id_col].astype(str)
    base["skill_uri"] = base["skill_uri"].astype(str)
    base["w"] = base["w"].astype(float)

    # dedupe keeping max weight per (id, skill)
    base = base.groupby([id_col, "skill_uri"], as_index=False)["w"].max()

    current = base.copy()
    all_rows = [base]

    for hop in range(1, int(hops) + 1):
        dec = float(neighbor_w) ** hop
        rows = []
        for _id, g in current.groupby(id_col, sort=False):
            for s, w in zip(g["skill_uri"].tolist(), g["w"].tolist()):
                neigh = adj.get(str(s), set())
                if not neigh:
                    continue
                for n in neigh:
                    rows.append({id_col: str(_id), "skill_uri": str(n), "w": float(w) * dec})
        if not rows:
            break
        nxt = pd.DataFrame(rows)
        nxt = nxt.groupby([id_col, "skill_uri"], as_index=False)["w"].max()
        all_rows.append(nxt)
        current = nxt

    out = pd.concat(all_rows, ignore_index=True)
    out = out.groupby([id_col, "skill_uri"], as_index=False)["w"].max()
    return out


doc_skills_aug = expand_pairs(doc_skills_df, "docno", hops=KG_EXPAND_HOPS, neighbor_w=KG_NEIGHBOR_W)
qid_skills_aug = expand_pairs(qid_skills_df, "qid", hops=KG_EXPAND_HOPS, neighbor_w=KG_NEIGHBOR_W)

print("aug doc skills:", doc_skills_aug.shape, "aug qid skills:", qid_skills_aug.shape)


# =============================================================================
# [CELL 7] Build ESCO triples and train ComplEx with PyKEEN
# =============================================================================

import torch
from pykeen.triples import TriplesFactory
from pykeen.pipeline import pipeline


def build_esco_triples(hier_path: str, rel_path: str) -> pd.DataFrame:
    hier = pd.read_csv(hier_path).fillna("")
    rel = pd.read_csv(rel_path).fillna("")
    triples: List[Tuple[str, str, str]] = []

    lvl_cols_local = [c for c in ["Level 0 URI", "Level 1 URI", "Level 2 URI", "Level 3 URI"] if c in hier.columns]
    for _, row in hier[lvl_cols_local].iterrows():
        uris = [str(row[c]).strip() for c in lvl_cols_local if str(row[c]).strip() and str(row[c]).strip() != "nan"]
        for a, b in zip(uris[:-1], uris[1:]):
            triples.append((a, "broader", b))
            triples.append((b, "narrower", a))

    if "relationType" in rel.columns:
        rel = rel[rel["relationType"].astype(str) == "optional"].copy()

    if "originalSkillUri" in rel.columns and "relatedSkillUri" in rel.columns:
        for a, b in zip(rel["originalSkillUri"].astype(str), rel["relatedSkillUri"].astype(str)):
            a = a.strip()
            b = b.strip()
            if a and b and a != "nan" and b != "nan":
                triples.append((a, "optional", b))
                triples.append((b, "optional_rev", a))

    return pd.DataFrame(triples, columns=["head", "relation", "tail"]).drop_duplicates()


triples_df = build_esco_triples(HIER_PATH, REL_PATH)
print("triples:", triples_df.shape, "relations:", triples_df["relation"].nunique())

# Train model (cached, robust)

COMPLEX_CACHE_DIR.mkdir(parents=True, exist_ok=True)

cache_model_path = COMPLEX_CACHE_DIR / f"complex_model_dim{COMPLEX_EMB_DIM}_ep{COMPLEX_EPOCHS}.pkl"
cache_tf_pkl_path = COMPLEX_CACHE_DIR / f"triples_factory_dim{COMPLEX_EMB_DIM}_ep{COMPLEX_EPOCHS}.pkl"

# Legacy path that may exist as a directory from older runs
legacy_tf_path = COMPLEX_CACHE_DIR / "triples_factory.tsv"
if legacy_tf_path.exists() and legacy_tf_path.is_dir():
    # Rename instead of deleting, to preserve anything inside
    legacy_backup = COMPLEX_CACHE_DIR / "triples_factory_legacy_dir_backup"
    try:
        if legacy_backup.exists():
            # If backup exists, append a suffix
            legacy_backup = COMPLEX_CACHE_DIR / "triples_factory_legacy_dir_backup_2"
        legacy_tf_path.rename(legacy_backup)
        print("Renamed legacy TriplesFactory directory to:", legacy_backup)
    except Exception as e:
        print("Warning: could not rename legacy TriplesFactory directory:", legacy_tf_path, "error:", e)

import pickle

if cache_model_path.exists() and cache_tf_pkl_path.exists():
    with open(cache_model_path, "rb") as f:
        complex_model = pickle.load(f)
    with open(cache_tf_pkl_path, "rb") as f:
        tf_for_vocab = pickle.load(f)
    print("Loaded cached ComplEx model and TriplesFactory")
else:
    triples = triples_df[["head", "relation", "tail"]].astype(str).values
    tf = TriplesFactory.from_labeled_triples(triples)
    train_tf, test_tf = tf.split(ratios=(0.9, 0.1), random_state=42)

    result = pipeline(
        training=train_tf,
        testing=test_tf,
        model="ComplEx",
        model_kwargs={"embedding_dim": int(COMPLEX_EMB_DIM)},
        training_kwargs={"num_epochs": int(COMPLEX_EPOCHS), "batch_size": int(COMPLEX_BATCH_SIZE)},
        optimizer="Adam",
        optimizer_kwargs={"lr": float(COMPLEX_LR)},
        random_seed=42,
        device=torch.device(COMPLEX_DEVICE),
    )
    complex_model = result.model
    tf_for_vocab = train_tf

    with open(cache_model_path, "wb") as f:
        pickle.dump(complex_model, f)
    with open(cache_tf_pkl_path, "wb") as f:
        pickle.dump(tf_for_vocab, f)

    print("Trained and cached ComplEx model and TriplesFactory")


# Train/load RotatE model (cached, robust)
rotate_model_path = COMPLEX_CACHE_DIR / f"rotate_model_dim{ROTATE_EMB_DIM}_ep{ROTATE_EPOCHS}.pkl"
rotate_tf_pkl_path = COMPLEX_CACHE_DIR / f"rotate_triples_factory_dim{ROTATE_EMB_DIM}_ep{ROTATE_EPOCHS}.pkl"

if rotate_model_path.exists() and rotate_tf_pkl_path.exists():
    with open(rotate_model_path, "rb") as f:
        rotate_model = pickle.load(f)
    with open(rotate_tf_pkl_path, "rb") as f:
        rotate_tf_for_vocab = pickle.load(f)
    print("Loaded cached RotatE model and TriplesFactory")
else:
    triples = triples_df[["head", "relation", "tail"]].astype(str).values
    tf_r = TriplesFactory.from_labeled_triples(triples)
    train_tf_r, test_tf_r = tf_r.split(ratios=(0.9, 0.1), random_state=42)

    rotate_result = pipeline(
        training=train_tf_r,
        testing=test_tf_r,
        model="RotatE",
        model_kwargs={"embedding_dim": int(ROTATE_EMB_DIM)},
        training_kwargs={"num_epochs": int(ROTATE_EPOCHS), "batch_size": int(ROTATE_BATCH_SIZE)},
        optimizer="Adam",
        optimizer_kwargs={"lr": float(ROTATE_LR)},
        random_seed=42,
        device=torch.device(ROTATE_DEVICE),
    )
    rotate_model = rotate_result.model
    rotate_tf_for_vocab = train_tf_r

    with open(rotate_model_path, "wb") as f:
        pickle.dump(rotate_model, f)
    with open(rotate_tf_pkl_path, "wb") as f:
        pickle.dump(rotate_tf_for_vocab, f)

    print("Trained and cached RotatE model and TriplesFactory")



def build_complex_skill_embeddings(model, tf_vocab: TriplesFactory, skill_uris: List[str], device: str) -> Tuple[Dict[str, np.ndarray], int]:
    model = model.to(device)
    model.eval()
    ent_to_id = tf_vocab.entity_to_id

    keep = [s for s in skill_uris if s in ent_to_id]
    if not keep:
        raise ValueError("No skill URIs from extraction are present in the ComplEx vocabulary")

    ids = torch.as_tensor([ent_to_id[s] for s in keep], dtype=torch.long, device=device)
    with torch.no_grad():
        rep = model.entity_representations[0](indices=ids)
        if torch.is_complex(rep):
            vec = torch.cat([rep.real, rep.imag], dim=1)
        else:
            vec = rep
        vec = vec.detach().cpu().numpy().astype(np.float32)

    return {s: v for s, v in zip(keep, vec)}, int(vec.shape[1])

def build_rotate_skill_embeddings(model, tf_vocab: TriplesFactory, skill_uris: List[str], device: str) -> Tuple[Dict[str, np.ndarray], int]:
    """Extract RotatE entity embeddings (real vectors)."""
    model = model.to(device)
    model.eval()
    ent_to_id = tf_vocab.entity_to_id

    keep = [s for s in skill_uris if s in ent_to_id]
    if not keep:
        raise ValueError("No skill URIs from extraction are present in the RotatE vocabulary")

    ids = torch.as_tensor([ent_to_id[s] for s in keep], dtype=torch.long, device=device)
    with torch.no_grad():
        rep = model.entity_representations[0](indices=ids)
        if torch.is_complex(rep):
            vec = torch.cat([rep.real, rep.imag], dim=1)
        else:
            vec = rep
        vec = vec.detach().cpu().numpy().astype(np.float32)

    return {s: v for s, v in zip(keep, vec)}, int(vec.shape[1])



skill_uris_used = pd.Index(pd.concat([
    doc_skills_aug["skill_uri"].astype(str),
    qid_skills_aug["skill_uri"].astype(str),
])).unique().tolist()

skill_uri_to_vec, d_complex = build_complex_skill_embeddings(
    complex_model,
    tf_for_vocab,
    skill_uris_used,
    device=COMPLEX_DEVICE,
)
print("ComplEx vectors:", len(skill_uri_to_vec), "dim:", d_complex)


def aggregate_vectors(skills_df_w: pd.DataFrame, id_col: str, d: int) -> Dict[str, np.ndarray]:
    zero = np.zeros(d, dtype=np.float32)
    out: Dict[str, np.ndarray] = {}

    if skills_df_w.empty:
        return out

    for _id, g in skills_df_w.groupby(id_col, sort=False):
        num = np.zeros(d, dtype=np.float32)
        den = 0.0
        for s, w in zip(g["skill_uri"].astype(str), g["w"].astype(float)):
            v = skill_uri_to_vec.get(s)
            if v is None:
                continue
            ww = float(w)
            num += ww * v
            den += ww
        v = (num / (den + 1e-12)).astype(np.float32)
        v = power_normalize(v, p=0.5)
        out[str(_id)] = l2_normalize(v) if den > 0 else zero.copy()

    return out


docno2kg = aggregate_vectors(doc_skills_aug, "docno", d_complex)
qid2kg = aggregate_vectors(qid_skills_aug, "qid", d_complex)

print("docno2kg:", len(docno2kg), "qid2kg:", len(qid2kg))


# Build RotatE skill vectors and aggregate
rotate_skill_uri_to_vec, d_rotate = build_rotate_skill_embeddings(
    rotate_model,
    rotate_tf_for_vocab,
    skill_uris_used,
    device=ROTATE_DEVICE,
)
print("RotatE vectors:", len(rotate_skill_uri_to_vec), "dim:", d_rotate)

def aggregate_vectors_rotate(skills_df_w: pd.DataFrame, id_col: str, d: int) -> Dict[str, np.ndarray]:
    zero = np.zeros(d, dtype=np.float32)
    out: Dict[str, np.ndarray] = {}
    if skills_df_w.empty:
        return out
    for _id, g in skills_df_w.groupby(id_col, sort=False):
        num = np.zeros(d, dtype=np.float32)
        den = 0.0
        for s, w in zip(g["skill_uri"].astype(str), g["w"].astype(float)):
            v = rotate_skill_uri_to_vec.get(s)
            if v is None:
                continue
            ww = float(w)
            num += ww * v
            den += ww
        v = (num / (den + 1e-12)).astype(np.float32)
        v = power_normalize(v, p=0.5)
        out[str(_id)] = l2_normalize(v) if den > 0 else zero.copy()
    return out

docno2kg_rotate = aggregate_vectors_rotate(doc_skills_aug, "docno", d_rotate)
qid2kg_rotate = aggregate_vectors_rotate(qid_skills_aug, "qid", d_rotate)

print("docno2kg_rotate:", len(docno2kg_rotate), "qid2kg_rotate:", len(qid2kg_rotate))


# =============================================================================
# [CELL 8] PyTerrier: index, dense retriever, and rerankers
# =============================================================================

import pyterrier as pt
from sentence_transformers import CrossEncoder

if not pt.java.started():
    pt.java.init()


def get_or_build_index(index_dir: Path, corpus_df: pd.DataFrame) -> pt.IndexRef:
    index_dir = Path(index_dir)
    if (index_dir / "data.properties").exists():
        return pt.IndexRef.of(str(index_dir))
    index_dir.mkdir(parents=True, exist_ok=True)
    indexer = pt.IterDictIndexer(str(index_dir), meta={"docno": 64}, text_attrs=["text"])
    return indexer.index(corpus_df[["docno", "text"]].to_dict("records"))


class FaissDenseRetriever(pt.Transformer):
    def __init__(self, corpus_df: pd.DataFrame, model_name: str, topk: int, show_progress: bool = True):
        super().__init__()
        self.topk = int(topk)

        cdf = corpus_df[["docno", "text"]].copy()
        cdf["docno"] = cdf["docno"].astype(str)
        cdf["text"] = cdf["text"].astype(str)
        self.docnos = cdf["docno"].tolist()

        self.st = SentenceTransformer(model_name)

        Xdoc = self.st.encode(
            cdf["text"].tolist(),
            convert_to_numpy=True,
            normalize_embeddings=True,
            show_progress_bar=bool(show_progress),
        ).astype("float32")

        self.index = faiss.IndexFlatIP(Xdoc.shape[1])
        self.index.add(Xdoc)

    def transform(self, topics_df: pd.DataFrame) -> pd.DataFrame:
        qids = topics_df["qid"].astype(str).tolist()
        qs = topics_df["query"].astype(str).tolist()
        Q = self.st.encode(qs, convert_to_numpy=True, normalize_embeddings=True, show_progress_bar=False).astype("float32")
        scores, idxs = self.index.search(Q, self.topk)

        rows = []
        for i, qid in enumerate(qids):
            for rank, (j, sc) in enumerate(zip(idxs[i], scores[i]), start=1):
                if j < 0:
                    continue
                rows.append({"qid": qid, "docno": self.docnos[j], "score": float(sc), "rank": int(rank)})
        return pd.DataFrame(rows)


class CrossEncoderReranker(pt.Transformer):
    def __init__(self, corpus_df: pd.DataFrame, model_name: str, batch_size: int = 32, max_length: int = 512):
        super().__init__()
        self.ce = CrossEncoder(model_name, max_length=int(max_length))
        self.batch_size = int(batch_size)
        self.docno2text = dict(zip(corpus_df["docno"].astype(str), corpus_df["text"].astype(str)))

    @staticmethod
    def to_1d(pred) -> np.ndarray:
        pred = np.asarray(pred)
        if pred.ndim == 1:
            return pred.astype(np.float32)
        if pred.ndim == 2 and pred.shape[1] == 2:
            x = pred.astype(np.float32)
            x = x - x.max(axis=1, keepdims=True)
            ex = np.exp(x)
            return (ex[:, 1] / (ex.sum(axis=1) + 1e-12)).astype(np.float32)
        return pred.squeeze().astype(np.float32)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = ensure_run(df)
        if "query" not in df.columns:
            raise ValueError("CrossEncoderReranker needs query column, merge topics first")
        df = df.copy()
        df["text"] = [self.docno2text.get(d, "") for d in df["docno"]]
        pairs = list(zip(df["query"].astype(str).tolist(), df["text"].astype(str).tolist()))
        pred = self.ce.predict(pairs, batch_size=self.batch_size, show_progress_bar=True)
        df["score"] = self.to_1d(pred).astype(float)
        df = df.sort_values(["qid", "score"], ascending=[True, False])
        df["rank"] = df.groupby("qid").cumcount() + 1
        return df


class KGReranker(pt.Transformer):
    def __init__(self, qid2vec: Dict[str, np.ndarray], docno2vec: Dict[str, np.ndarray], alpha: float = 0.0):
        super().__init__()
        self.qid2vec = qid2vec
        self.docno2vec = docno2vec
        self.alpha = float(alpha)
        self.dim = int(next(iter(docno2vec.values())).shape[0]) if docno2vec else int(next(iter(qid2vec.values())).shape[0])
        self.zero = np.zeros(self.dim, dtype=np.float32)

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = ensure_run(df)
        df = df.copy()
        df["orig_score"] = df["score"].astype(float)

        Q = np.vstack([self.qid2vec.get(q, self.zero) for q in df["qid"]]).astype(np.float32)
        D = np.vstack([self.docno2vec.get(d, self.zero) for d in df["docno"]]).astype(np.float32)

        kg = (l2_normalize_rows(Q) * l2_normalize_rows(D)).sum(axis=1).astype(np.float32)
        df["kg_score"] = kg.astype(float)

        a = self.alpha
        df["score"] = ((1.0 - a) * df["kg_score"] + a * df["orig_score"]).astype(float)

        df = df.sort_values(["qid", "score"], ascending=[True, False])
        df["rank"] = df.groupby("qid").cumcount() + 1
        return df


def cut_k(k: int) -> pt.Transformer:
    return pt.apply.generic(lambda df: df.groupby("qid", sort=False).head(int(k)))


def add_query_col(topics_df: pd.DataFrame) -> pt.Transformer:
    tq = topics_df[["qid", "query"]].copy()
    tq["qid"] = tq["qid"].astype(str)

    def _merge(df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df["qid"] = df["qid"].astype(str)
        return df.merge(tq, on="qid", how="left")

    return pt.apply.generic(_merge)


index_ref = get_or_build_index(INDEX_DIR, corpus)
dense100 = FaissDenseRetriever(corpus, model_name=DENSE_MODEL, topk=K_CAND)
add_query = add_query_col(topics)
cut50 = cut_k(K_FINAL)

ce = CrossEncoderReranker(corpus, model_name=CE_MODEL, batch_size=32)

lex_bm25 = pt.BatchRetrieve(index_ref, wmodel="BM25")
lex_dph = pt.BatchRetrieve(index_ref, wmodel="DPH")
lex_pl2 = pt.BatchRetrieve(index_ref, wmodel="PL2")
lex_tfidf = pt.BatchRetrieve(index_ref, wmodel="TF_IDF")

# KG rerank: alpha=0 means pure KG, consider alpha 0.2 to 0.6 if KG is noisy
KG_ALPHA = 0.5  # blend weight on original dense score; try 0.2, 0.4, 0.6
kg = KGReranker(qid2kg, docno2kg, alpha=KG_ALPHA)
kg_rotate = KGReranker(qid2kg_rotate, docno2kg_rotate, alpha=KG_ALPHA)


# =============================================================================
# [CELL 9] Pipelines (requested set only)
# =============================================================================

all_pipelines: List[Tuple[str, pt.Transformer]] = [
    ("dense@100>>CrossEncoder->@50", dense100 >> add_query >> ce >> cut50),
    ("dense@100>>BM25(rescore)->@50", dense100 >> add_query >> lex_bm25 >> cut50),
    ("dense@100>>DPH(rescore)->@50", dense100 >> add_query >> lex_dph >> cut50),
    ("dense@100>>PL2(rescore)->@50", dense100 >> add_query >> lex_pl2 >> cut50),
    ("dense@100>>TF_IDF(rescore)->@50", dense100 >> add_query >> lex_tfidf >> cut50),
    (f"dense@100>>KG(ComplEx,alpha={KG_ALPHA})->@50", dense100 >> kg >> cut50),
    (f"dense@100>>KG(RotatE,alpha={KG_ALPHA})->@50", dense100 >> kg_rotate >> cut50),
]

print("pipelines:", [n for n, _ in all_pipelines])


# =============================================================================
# [CELL 10] Evaluate with PyTerrier Experiment
# =============================================================================

def eval_at_k(pipelines: List[Tuple[str, pt.Transformer]], topics_df: pd.DataFrame, qrels_path: str, k_eval: int) -> pd.DataFrame:
    metrics = [
        f"ndcg_cut.{k_eval}",
        "recip_rank",
        f"P.{k_eval}",
        f"recall.{k_eval}",
        f"map_cut.{k_eval}",
    ]
    df = pt.Experiment(
        [p for _, p in pipelines],
        topics_df,
        qrels_path,
        eval_metrics=metrics,
        names=[n for n, _ in pipelines],
        verbose=True,
        validate="ignore",
    )

    df = df.rename(columns={
        f"ndcg_cut.{k_eval}": f"nDCG@{k_eval}",
        "recip_rank": f"MRR@{k_eval}",
        f"P.{k_eval}": f"Prec@{k_eval}",
        f"recall.{k_eval}": f"Recall@{k_eval}",
        f"map_cut.{k_eval}": f"MAP@{k_eval}",
    }).copy()

    p = df[f"Prec@{k_eval}"].astype(float)
    r = df[f"Recall@{k_eval}"].astype(float)
    df[f"F1@{k_eval}"] = np.where((p + r) > 0, 2 * p * r / (p + r), 0.0)

    cols = ["name", f"nDCG@{k_eval}", f"MRR@{k_eval}", f"Prec@{k_eval}", f"Recall@{k_eval}", f"F1@{k_eval}", f"MAP@{k_eval}"]
    return df[cols]


metrics_df = eval_at_k(all_pipelines, topics, QRELS_PATH, k_eval=K_FINAL).sort_values(f"nDCG@{K_FINAL}", ascending=False)
print(metrics_df.to_string(index=False))




cwd: /Users/user/Projects/question-retrieval-KG
INDEX_DIR: /Users/user/Projects/question-retrieval-KG/var/artifacts/kg_reranker_v3_refactored/terrier_index
corpus: (2812, 2) topics: (15, 2)
ESCO skills: 13939 adj nodes: 6173


/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|███████████████████████████████████████████████| 436/436 [00:12<00:00, 34.04it/s]


skills indexed: 13939 dim: 768
doc_skills_df: (40997, 3) unique docs: 2800
qid_skills_df: (300, 3) unique qids: 15
aug doc skills: (64960, 3) aug qid skills: (500, 3)
triples: (12530, 3) relations: 4
Loaded cached ComplEx model and TriplesFactory
Loaded cached RotatE model and TriplesFactory
ComplEx vectors: 3570 dim: 1024
docno2kg: 2800 qid2kg: 15
RotatE vectors: 3570 dim: 512
docno2kg_rotate: 2800 qid2kg_rotate: 15


Java started and loaded: pyterrier.java.colab, pyterrier.java, pyterrier.java.24, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Batches: 100%|█████████████████████████████████████████████████| 88/88 [00:29<00:00,  2.95it/s]
/Users/user/miniconda3/envs/KG-re-ranking/lib/python3.11/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/var/folders/2g/5qz9ypl17kx1hgsgxs5rx7v80000gn/T/ipykernel_32440/2751915344.py:757: Deprecat

pipelines: ['dense@100>>CrossEncoder->@50', 'dense@100>>BM25(rescore)->@50', 'dense@100>>DPH(rescore)->@50', 'dense@100>>PL2(rescore)->@50', 'dense@100>>TF_IDF(rescore)->@50', 'dense@100>>KG(ComplEx,alpha=0.5)->@50', 'dense@100>>KG(RotatE,alpha=0.5)->@50']


pt.Experiment: 100%|█████████████████████████████████████████| 7/7 [00:03<00:00,  1.87system/s]

                                 name  nDCG@50   MRR@50  Prec@50  Recall@50    F1@50   MAP@50
dense@100>>KG(ComplEx,alpha=0.5)->@50 0.551581 0.792398 0.512000   0.444103 0.475641 0.318823
 dense@100>>KG(RotatE,alpha=0.5)->@50 0.549747 0.837255 0.510667   0.435331 0.469999 0.320912
         dense@100>>CrossEncoder->@50 0.465119 0.681058 0.450667   0.397329 0.422320 0.236662
         dense@100>>DPH(rescore)->@50 0.362540 0.719444 0.381333   0.279414 0.322513 0.155366
      dense@100>>TF_IDF(rescore)->@50 0.357032 0.669444 0.381333   0.279250 0.322404 0.152819
         dense@100>>PL2(rescore)->@50 0.353791 0.669444 0.378667   0.277562 0.320326 0.150033
        dense@100>>BM25(rescore)->@50 0.317927 0.613586 0.356000   0.243017 0.288854 0.135366


In [2]:
metrics_df

,name,nDCG@50,MRR@50,Prec@50,Recall@50,F1@50,MAP@50
5,"dense@100>>KG(ComplEx,alpha=0.5)->@50",0.551581,0.792398,0.512000,0.444103,0.475641,0.318823
6,"dense@100>>KG(RotatE,alpha=0.5)->@50",0.549747,0.837255,0.510667,0.435331,0.469999,0.320912
0,dense@100>>CrossEncoder->@50,0.465119,0.681058,0.450667,0.397329,0.422320,0.236662
2,dense@100>>DPH(rescore)->@50,0.362540,0.719444,0.381333,0.279414,0.322513,0.155366
4,dense@100>>TF_IDF(rescore)->@50,0.357032,0.669444,0.381333,0.279250,0.322404,0.152819
3,dense@100>>PL2(rescore)->@50,0.353791,0.669444,0.378667,0.277562,0.320326,0.150033
1,dense@100>>BM25(rescore)->@50,0.317927,0.613586,0.356000,0.243017,0.288854,0.135366


In [3]:
# Avoid duplicates before appending (keeps the first occurrence)
def _dedup_pipelines(pipes):
    seen = set()
    out = []
    for name, pipe in pipes:
        if name in seen:
            continue
        seen.add(name)
        out.append((name, pipe))
    return out

# remove any previous accidental duplicates of these names
all_pipelines = [(n, p) for (n, p) in all_pipelines if n not in {"dense@100->@50", "dense@100(trim)->@50"}]

# add only one dense-only variant (use the notebook's cut50 transformer)
all_pipelines.append(("dense@100->@50", dense100 >> cut50))

# (optional) safety dedup in case something else duplicated names
all_pipelines = _dedup_pipelines(all_pipelines)

# re-evaluate
metrics_df = eval_at_k(all_pipelines, topics, QRELS_PATH, k_eval=50)
metrics_df = metrics_df.sort_values("nDCG@50", ascending=False).reset_index(drop=True)
metrics_df

pt.Experiment: 100%|█████████████████████████████████████████| 8/8 [00:03<00:00,  2.47system/s]


,name,nDCG@50,MRR@50,Prec@50,Recall@50,F1@50,MAP@50
0,"dense@100>>KG(ComplEx,alpha=0.5)->@50",0.551581,0.792398,0.512000,0.444103,0.475641,0.318823
1,"dense@100>>KG(RotatE,alpha=0.5)->@50",0.549747,0.837255,0.510667,0.435331,0.469999,0.320912
2,dense@100->@50,0.500596,0.794444,0.462667,0.411309,0.435479,0.273427
3,dense@100>>CrossEncoder->@50,0.465119,0.681058,0.450667,0.397329,0.422320,0.236662
4,dense@100>>DPH(rescore)->@50,0.362540,0.719444,0.381333,0.279414,0.322513,0.155366
5,dense@100>>TF_IDF(rescore)->@50,0.357032,0.669444,0.381333,0.279250,0.322404,0.152819
6,dense@100>>PL2(rescore)->@50,0.353791,0.669444,0.378667,0.277562,0.320326,0.150033
7,dense@100>>BM25(rescore)->@50,0.317927,0.613586,0.356000,0.243017,0.288854,0.135366


In [4]:
metrics_df.round(2)

,name,nDCG@50,MRR@50,Prec@50,Recall@50,F1@50,MAP@50
0,"dense@100>>KG(ComplEx,alpha=0.5)->@50",0.55,0.79,0.51,0.44,0.48,0.32
1,"dense@100>>KG(RotatE,alpha=0.5)->@50",0.55,0.84,0.51,0.44,0.47,0.32
2,dense@100->@50,0.50,0.79,0.46,0.41,0.44,0.27
3,dense@100>>CrossEncoder->@50,0.47,0.68,0.45,0.40,0.42,0.24
4,dense@100>>DPH(rescore)->@50,0.36,0.72,0.38,0.28,0.32,0.16
5,dense@100>>TF_IDF(rescore)->@50,0.36,0.67,0.38,0.28,0.32,0.15
6,dense@100>>PL2(rescore)->@50,0.35,0.67,0.38,0.28,0.32,0.15
7,dense@100>>BM25(rescore)->@50,0.32,0.61,0.36,0.24,0.29,0.14


In [5]:
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import normalize
from scipy.sparse import csr_matrix


# SVD settings (start here)
SVD_DIM = 256
RANDOM_STATE = 42

svd_skill_vocab = pd.Index(pd.concat([
    doc_skills_aug["skill_uri"].astype(str),
    qid_skills_aug["skill_uri"].astype(str),
])).unique().tolist()

skill2idx_svd = {u:i for i,u in enumerate(svd_skill_vocab)}
n_sk = len(svd_skill_vocab)

doc_ids = corpus["docno"].astype(str).unique().tolist()
qid_ids = topics["qid"].astype(str).unique().tolist()
doc2row = {d:i for i,d in enumerate(doc_ids)}
qid2row = {q:i for i,q in enumerate(qid_ids)}

def to_sparse_svd(df, id_col, id2row):
    if df.empty:
        return csr_matrix((len(id2row), n_sk), dtype=np.float32)
    r = df[id_col].astype(str).map(id2row).to_numpy()
    c = df["skill_uri"].astype(str).map(skill2idx_svd).to_numpy()
    v = df["w"].astype(float).to_numpy()
    m = csr_matrix((v, (r, c)), shape=(len(id2row), n_sk), dtype=np.float32)
    return m

X_doc = to_sparse_svd(doc_skills_aug, "docno", doc2row)
X_qid = to_sparse_svd(qid_skills_aug, "qid", qid2row)

svd = TruncatedSVD(n_components=SVD_DIM, random_state=RANDOM_STATE)
E_doc = svd.fit_transform(X_doc)
E_qid = svd.transform(X_qid)

E_doc = normalize(E_doc)
E_qid = normalize(E_qid)

docno2kg_svd = {doc_ids[i]: E_doc[i].astype(np.float32) for i in range(len(doc_ids))}
qid2kg_svd   = {qid_ids[i]: E_qid[i].astype(np.float32) for i in range(len(qid_ids))}

print("SVD explained_var:", float(svd.explained_variance_ratio_.sum()))
print("docno2kg_svd:", len(docno2kg_svd), "qid2kg_svd:", len(qid2kg_svd))


kg_svd = KGReranker(qid2kg_svd, docno2kg_svd, alpha=0.5)

# Add SVD KG reranker as a new experiment in the existing list
all_pipelines.append(
    ("dense@100>>KG(SVD,alpha=0.5)->@50", dense100 >> kg_svd >> cut50)
)


print("pipelines:", [n for n, _ in all_pipelines])



SVD explained_var: 0.6500351428985596
docno2kg_svd: 2812 qid2kg_svd: 15
pipelines: ['dense@100>>CrossEncoder->@50', 'dense@100>>BM25(rescore)->@50', 'dense@100>>DPH(rescore)->@50', 'dense@100>>PL2(rescore)->@50', 'dense@100>>TF_IDF(rescore)->@50', 'dense@100>>KG(ComplEx,alpha=0.5)->@50', 'dense@100>>KG(RotatE,alpha=0.5)->@50', 'dense@100->@50', 'dense@100>>KG(SVD,alpha=0.5)->@50']


In [6]:
metrics_df = eval_at_k(all_pipelines, topics, QRELS_PATH, k_eval=K_FINAL)\
    .sort_values(f"nDCG@{K_FINAL}", ascending=False)\
    .reset_index(drop=True)

metrics_df.round(3)

pt.Experiment: 100%|█████████████████████████████████████████| 9/9 [00:03<00:00,  2.75system/s]


,name,nDCG@50,MRR@50,Prec@50,Recall@50,F1@50,MAP@50
0,"dense@100>>KG(SVD,alpha=0.5)->@50",0.599,0.887,0.555,0.468,0.507,0.358
1,"dense@100>>KG(ComplEx,alpha=0.5)->@50",0.552,0.792,0.512,0.444,0.476,0.319
2,"dense@100>>KG(RotatE,alpha=0.5)->@50",0.550,0.837,0.511,0.435,0.470,0.321
3,dense@100->@50,0.501,0.794,0.463,0.411,0.435,0.273
4,dense@100>>CrossEncoder->@50,0.465,0.681,0.451,0.397,0.422,0.237
5,dense@100>>DPH(rescore)->@50,0.363,0.719,0.381,0.279,0.323,0.155
6,dense@100>>TF_IDF(rescore)->@50,0.357,0.669,0.381,0.279,0.322,0.153
7,dense@100>>PL2(rescore)->@50,0.354,0.669,0.379,0.278,0.320,0.150
8,dense@100>>BM25(rescore)->@50,0.318,0.614,0.356,0.243,0.289,0.135


In [7]:


from ranx import Qrels, Run, evaluate
from scipy.stats import ttest_rel, wilcoxon

K = int(K_FINAL)
metric = f"ndcg@{K}"

def df_to_ranx_run(df: pd.DataFrame, name: str) -> Run:
    df = df.copy()
    df["qid"] = df["qid"].astype(str)
    df["docno"] = df["docno"].astype(str)
    df["score"] = df["score"].astype(float)

    run_dict = {}
    for qid, g in df.groupby("qid", sort=False):
        gg = g.groupby("docno", as_index=False)["score"].max().sort_values("score", ascending=False)
        run_dict[qid] = dict(zip(gg["docno"].tolist(), gg["score"].tolist()))
    return Run(run_dict, name=name)

def build_runs_from_pipelines(pipelines, topics_df: pd.DataFrame) -> dict:
    runs = {}
    for name, pipe in pipelines:
        out = pipe.transform(topics_df)
        if out is None or len(out) == 0:
            raise RuntimeError(f"Pipeline returned empty output: {name}")
        need = {"qid", "docno", "score"}
        missing = need - set(out.columns)
        if missing:
            raise ValueError(f"Missing columns {missing} in pipeline output for: {name}")
        runs[name] = df_to_ranx_run(out[["qid", "docno", "score"]], name=name)
    return runs

# Load qrels
qrels = Qrels.from_file(QRELS_PATH, kind="trec")

# Build runs
runs = build_runs_from_pipelines(all_pipelines, topics)

# --------- Per-query evaluation (API compatible) ----------
def single_qrels(qrels: Qrels, qid: str) -> Qrels:
    """
    Create a Qrels object containing only one query.
    Works with ranx by passing a dict {qid: {docid: rel}}.
    """
    # Qrels stores a dict-like .qrels in most versions
    qrels_dict = getattr(qrels, "qrels", None)
    if qrels_dict is None:
        # Fallback: convert via to_dict if available
        if hasattr(qrels, "to_dict"):
            qrels_dict = qrels.to_dict()
        else:
            raise RuntimeError("Cannot access qrels dict from your ranx version.")
    if qid not in qrels_dict:
        return Qrels({qid: {}}, name=getattr(qrels, "name", "qrels"))
    return Qrels({qid: qrels_dict[qid]}, name=getattr(qrels, "name", "qrels"))

def single_run(run: Run, qid: str) -> Run:
    run_dict = getattr(run, "run", None)
    if run_dict is None:
        if hasattr(run, "to_dict"):
            run_dict = run.to_dict()
        else:
            raise RuntimeError("Cannot access run dict from your ranx version.")
    if qid not in run_dict:
        return Run({qid: {}}, name=getattr(run, "name", "run"))
    return Run({qid: run_dict[qid]}, name=getattr(run, "name", "run"))

# Collect union of qids (from qrels is safest)
qrels_dict = getattr(qrels, "qrels", None)
if qrels_dict is None and hasattr(qrels, "to_dict"):
    qrels_dict = qrels.to_dict()
all_qids = sorted(list(qrels_dict.keys()))

per_query = {}
for name, run in runs.items():
    scores = {}
    for qid in all_qids:
        q_qrels = single_qrels(qrels, qid)
        q_run = single_run(run, qid)
        # evaluate returns a dict {metric: value} for the single query case
        # --- ranx version safe: evaluate may return float or dict ---
        res = evaluate(q_qrels, q_run, metrics=[metric])

        if isinstance(res, (float, np.floating)):
            val = float(res)
        elif isinstance(res, dict):
            # usual newer style: {"ndcg@50": value}
            if metric in res:
                val = float(res[metric])
            # fallback: single metric dict
            elif len(res) == 1:
                val = float(next(iter(res.values())))
            else:
                raise ValueError(f"Unexpected dict result from ranx.evaluate: {res.keys()}")
        else:
            raise TypeError(f"Unexpected result type from ranx.evaluate: {type(res)}")

        scores[qid] = val
    per_query[name] = pd.Series(scores, dtype=float)

per_query_df = pd.DataFrame(per_query).sort_index()
display(per_query_df)

# --------- Significance tests ----------
def randomization_sign_flip_test(x: np.ndarray, y: np.ndarray, n_iters: int = 20000, seed: int = 42) -> float:
    rng = np.random.default_rng(seed)
    d = (x - y).astype(float)
    obs = float(np.mean(d))
    signs = rng.choice([-1.0, 1.0], size=(n_iters, d.size))
    samples = (signs * d[None, :]).mean(axis=1)
    p = (np.sum(np.abs(samples) >= abs(obs)) + 1.0) / (n_iters + 1.0)
    return float(p)

def paired_tests(system_a: str, system_b: str) -> pd.Series:
    a = per_query_df[system_a].to_numpy(dtype=float)
    b = per_query_df[system_b].to_numpy(dtype=float)
    diff = a - b

    t_p = float(ttest_rel(a, b).pvalue)

    if np.allclose(diff, 0.0):
        w_p = 1.0
    else:
        w_p = float(wilcoxon(diff, zero_method="wilcox", alternative="two-sided").pvalue)

    r_p = randomization_sign_flip_test(a, b, n_iters=20000, seed=42)

    return pd.Series({
        "system_a": system_a,
        "system_b": system_b,
        f"mean_{metric}_a": float(np.mean(a)),
        f"mean_{metric}_b": float(np.mean(b)),
        "mean_diff(a-b)": float(np.mean(diff)),
        "p_ttest": t_p,
        "p_wilcoxon": w_p,
        "p_randomization": r_p,
    })

# Pick best by mean per-query and compare to baselines (edit names if needed)
best = per_query_df.mean(axis=0).sort_values(ascending=False).index[0]

baseline_dense = "dense@100->@50"
baseline_ce = "dense@100>>CrossEncoder->@50"

results = pd.DataFrame([
    paired_tests(best, baseline_dense),
    paired_tests(best, baseline_ce),
]).reset_index(drop=True)

display(results)

print("Best system by mean per-query:", best)
print("Metric:", metric)

Batches: 100%|█████████████████████████████████████████████████| 47/47 [00:11<00:00,  4.01it/s]


,dense@100>>CrossEncoder->@50,dense@100>>BM25(rescore)->@50,dense@100>>DPH(rescore)->@50,dense@100>>PL2(rescore)->@50,dense@100>>TF_IDF(rescore)->@50,"dense@100>>KG(ComplEx,alpha=0.5)->@50","dense@100>>KG(RotatE,alpha=0.5)->@50",dense@100->@50,"dense@100>>KG(SVD,alpha=0.5)->@50"
0,0.729210,0.552808,0.548408,0.552472,0.552808,0.811714,0.834118,0.713171,0.878404
1,0.461003,0.651302,0.651302,0.651302,0.651302,0.655538,0.648057,0.490028,0.668361
10,0.651982,0.275908,0.298371,0.275908,0.275908,0.556105,0.572427,0.570507,0.838414
11,0.712988,0.655731,0.749354,0.709812,0.720598,0.838178,0.835410,0.717861,0.809422
12,0.487252,0.203409,0.284060,0.271560,0.271735,0.602190,0.598171,0.561072,0.618210
13,0.064743,0.000000,0.171521,0.171521,0.171521,0.103117,0.060659,0.172009,0.057588
14,0.500000,0.000000,0.000000,0.000000,0.000000,0.630930,0.630930,0.630930,0.430677
2,0.672620,0.314911,0.440055,0.417316,0.432929,0.598966,0.651818,0.498658,0.608440
3,0.392007,0.260425,0.260425,0.260425,0.260425,0.480632,0.428862,0.319099,0.498056
4,0.207033,0.408560,0.372537,0.370483,0.374326,0.313711,0.312246,0.281326,0.478191


,system_a,system_b,mean_ndcg@50_a,mean_ndcg@50_b,mean_diff(a-b),p_ttest,p_wilcoxon,p_randomization
0,"dense@100>>KG(SVD,alpha=0.5)->@50",dense@100->@50,0.598634,0.500596,0.098038,0.007521,0.025574,0.0097
1,"dense@100>>KG(SVD,alpha=0.5)->@50",dense@100>>CrossEncoder->@50,0.598634,0.465119,0.133514,0.000693,0.001526,0.0010


Best system by mean per-query: dense@100>>KG(SVD,alpha=0.5)->@50
Metric: ndcg@50
